# Exercises XP Ninja — A simple neural network with Keras

In this exercise, I build a neural network with Keras that predicts if an employee leaves the company, using the HR dataset from Kaggle. I encode the text columns, scale the features, split the data, create the model with `Sequential()`, train it, and evaluate its loss and accuracy.

### What I learn here
- How to prepare a table of data for a neural network: label encoding and feature scaling.
- How to stack layers in a Keras `Sequential` model.
- How to compile a model with a loss function and an optimizer, and train it.
- How to read the loss and the accuracy of a trained model.

## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. I download the dataset from [Kaggle (liujiaqi/hr-comma-sepcsv)](https://www.kaggle.com/datasets/liujiaqi/hr-comma-sepcsv) with the **Download** button. Kaggle gives a zip named `archive.zip`. I rename it to `HR_comma_sep.zip` and keep it in my `C:\Users\sivan\Datasets` folder.
3. I run the cells from top to bottom. When the upload cell opens a file picker, I choose `HR_comma_sep.zip`.

In [ ]:
from google.colab import files
# I import Colab's tool for uploading files from my computer.
uploaded = files.upload()
# I open a file picker and choose "HR_comma_sep.zip".
!unzip -o "HR_comma_sep.zip"
# I extract HR_comma_sep.csv from the zip into Colab.

## Exercise 1 : Create a simple Convolutionnal Neural networks (CNN)

**Instructions**

First read about this lesson about CNN : [Here](https://www.geeksforgeeks.org/convolutional-neural-network-cnn-in-machine-learning/)
and use this tutorial to do this exercise : [tutorial](https://medium.com/analytics-vidhya/deep-learning-tutorial-with-keras-7a34a1a322cd)

- Import required libraries including Keras
- Load this dataset (https://www.kaggle.com/datasets/liujiaqi/hr-comma-sepcsv)
- encode the label and scale the feature
- Split the dataset
- Create Neural Network Model using .sequential()
- Compile and Train model
- Evaluate model’s loss and accuracy

**A note on "CNN":** The lesson explains that convolutional layers are made for images, where nearby pixels belong together. The tutorial says the same: "a convolutional layer is best suited for data that contain images". The HR dataset is a table of 9 features about each employee, with no pixels and no order between the columns, so I build the network the steps ask for: a `Sequential` model of `Dense` layers.

## Step 1 — Import required libraries including Keras

In this step, I import pandas, the scikit-learn tools for preparing the data, and Keras from TensorFlow.

### What I learn here
- Keras is the simple, high-level part of TensorFlow for building neural networks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Keras comes inside TensorFlow, which is already installed in Colab.
from tensorflow import keras
from tensorflow.keras import layers

# I fix the random seed so the starting weights and the split are the same every run.
keras.utils.set_random_seed(42)

## Step 2 — Load this dataset

In this step, I load `HR_comma_sep.csv` and look at its size, first rows, column types and missing values. The column `left` is my target: 1 means the employee left the company and 0 means they stayed.

### What I learn here
- I check the data before I prepare it.

In [ ]:
# I read the CSV file that I extracted from the zip.
hr = pd.read_csv('HR_comma_sep.csv')
print(hr.shape)
display(hr.head())
print(hr.dtypes)
print('Missing values in the whole table:', hr.isna().sum().sum())
# I count the employees who stayed (0) and who left (1).
print(hr['left'].value_counts())

**What I see:** The dataset has **14,999 employees** and 10 columns, with **0 missing values**. Two columns are text: `Department` and `salary`. **11,428 employees stayed** (0) and **3,571 left** (1), so about 24% left. (The column name `average_montly_hours` is spelled like this in the dataset.)

## Step 3 — Encode the label and scale the feature

In this step, I turn the two text columns, `Department` and `salary`, into numbers with `LabelEncoder`, and I separate the features `X` from the target `y`. I create a `StandardScaler` for the features. I fit the scaler in Step 4, right after the split, on the training rows only, so the test rows stay unseen.

### What I learn here
- A neural network only reads numbers, so every text value must become a number.
- `LabelEncoder` gives every category a number: for `salary`, high = 0, low = 1, medium = 2 (alphabetical order).
- Scaling puts all features on the same scale (mean 0, std 1), which helps the network learn faster and more steadily.

In [ ]:
# I encode the text columns with one LabelEncoder per column.
for col in ['Department', 'salary']:
    encoder = LabelEncoder()
    hr[col] = encoder.fit_transform(hr[col])
    # classes_ lists the categories in the order of their numbers (0, 1, 2, ...).
    print(col, '->', dict(zip(encoder.classes_, range(len(encoder.classes_)))))

# X holds the features (all columns except 'left'), y holds the answer.
X = hr.drop(columns=['left'])
y = hr['left']

# I create the scaler: (value - training mean) / training standard deviation.
scaler = StandardScaler()
print(X.shape, y.shape)

## Step 4 — Split the dataset

In this step, I keep 20% of the employees for testing, with the same share of leavers in both sets. Then I fit the scaler on the training set and use it to scale both sets.

### What I learn here
- The model never sees the test rows during training, so the test score shows how it works on new employees.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# The scaler learns the means and standard deviations from the training rows only...
X_train = scaler.fit_transform(X_train)
# ...and uses the same numbers to scale the test rows.
X_test = scaler.transform(X_test)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Means after scaling (about 0):', X_train.mean(axis=0).round(2))

## Step 5 — Create Neural Network Model using .sequential()

In this step, I stack the layers of my network with `keras.Sequential()` (in Keras the name starts with a capital S).

### What I learn here
- The input layer receives the 9 features of one employee.
- The hidden `Dense` layers learn patterns. `Dense` means every neuron is connected to every input. `relu` keeps positive values and changes negative values to 0.
- The output layer has 1 neuron with `sigmoid`, which gives a probability between 0 and 1 that the employee leaves.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),   # 9 input features
    layers.Dense(32, activation='relu'),      # hidden layer 1: 32 neurons
    layers.Dense(16, activation='relu'),      # hidden layer 2: 16 neurons
    layers.Dense(1, activation='sigmoid'),    # output: probability of leaving
])
# summary() shows every layer, its output shape and its number of weights (parameters).
model.summary()

**My model:** The summary shows 3 `Dense` layers and **865 weights** to learn: 320 in the first layer (9 inputs × 32 neurons + 32 biases), 528 in the second (32 × 16 + 16) and 17 in the output layer (16 × 1 + 1).

## Step 6 — Compile and Train model

In this step, I compile the model with a loss function, an optimizer and a metric, and I train it.

### What I learn here
- `loss='binary_crossentropy'` is the loss for a yes/no problem: it is small when the predicted probability is close to the true answer.
- `optimizer='adam'` changes the weights a little after every batch to make the loss smaller.
- `epochs=30` means the network goes through the whole training set 30 times, and `batch_size=32` means it updates the weights after every 32 employees.
- `validation_split=0.2` keeps 20% of the training rows aside to check the model after every epoch.

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# fit() trains the network and returns its history: the loss and accuracy after every epoch.
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=2)

# I plot the accuracy and the loss of every epoch, for training and validation.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='validation')
axes[0].set_title('Accuracy per epoch'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='validation')
axes[1].set_title('Loss per epoch'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

**What I see:** In the first epochs, the accuracy goes up and the loss goes down quickly, then both slowly level off. After 30 epochs, the training accuracy is about **0.97** and the validation accuracy about **0.97** too. The validation curves stay close to the training curves, so the model is not overfitting.

## Step 7 — Evaluate model’s loss and accuracy

In this step, I evaluate my trained model on the test set, which it has never seen.

### What I learn here
- `evaluate()` returns the loss and the accuracy on new data.
- I compare the accuracy with always guessing "stayed", to see if the model really learned something.

In [ ]:
# evaluate() returns the loss and every metric from compile(), here [loss, accuracy].
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print('Test loss:', round(test_loss, 4))
print('Test accuracy:', round(test_accuracy, 4))

# A model that always says "stayed" (0) would be right for every employee who stayed.
print('Always guessing "stayed" would give:', round((y_test == 0).mean(), 4))

**My result:** On the test set, the loss is about **0.105** and the accuracy about **0.969**: the model correctly predicts whether about 97% of the 3,000 test employees leave. Always guessing "stayed" would give only **0.762**, so the network really learned the patterns. The numbers can change a little from run to run, because training a neural network has some randomness.